# **Identifying Argument Structure in John Searle's Texts Using NLI**
---
Authored by: Sangmin Seo

This notebook runs the full pipeline from `main.py` as a Google Colab notebook with GPU support. It reuses the modularized Python files from the [NLIstruct repository](https://github.com/seo-sangmin/NLIstruct).

## 1. Introduction
Identifying the structure of an argument within a text is a challenging yet crucial task across various fields, including philosophy. Given that an argument is an expression of inferences, the remarkable benchmark accuracy achieved in machine-based Natural Language Inference (NLI) tasks—thanks to the rapid development of Large Language Models (LLMs)—is noteworthy. However, despite the similarities between natural language argument and NLI, there is a lack of research that links the two fields.

### Research Question and Claims
In this Python notebook, the central question posed is: **Can NLI identify the argument structure in 'Searle (1980)' and 'Searle (2001)' in Wilson et al. (Eds.) (2001)?** I demonstrate that **even a model with a high benchmark score fails to label the argument types in the texts correctly.** Furthermore, even if we assume the existence of a model that labels correctly, **additional techniques and justifications are needed to reliably identify argument structures in the texts.** I will argue the points in Section 4 based on the results of the other sections.

### Data
To support my claims, I will use natural language texts **'Searle (1980)'** and **'Searle (2001)'**, both authored by John Searle. For the dataset, I will also employ the **SNLI corpus** to fine-tune BERT for natural language inference.

The choice of these texts may intrigue some readers, especially given that the Chinese Room argument presented in both works contends that programs do not understand natural language. However, the aim of this study is not to defend or critique the assertions made in these texts. Instead, the works will serve as test cases to evaluate NLI's capability to identify their argumentative structures.

### Techniques
Before diving into the analysis, I will examine the two texts to understand the situational contexts of their arguments. This preparatory work will facilitate a more informed evaluation of **NLI's** capabilities in argument analysis. To prepare the texts for analysis, I will undertake several tasks, including **pre-processing, exploratory analysis, topic modeling, sentence embedding, and text summarization**.

The **fine-tuning** of the model on the SNLI corpus, discussed in Section 3.1, takes approximately 25–50 minutes when using the T4 GPU on Google Colab. In Section 3.2, I will employ a different, pre-trained model for NLI tasks, so Section 3.1 is **optional** and can be skipped.

## Setup

**This notebook requires a GPU runtime.** In Colab, go to **Runtime → Change runtime type → T4 GPU → Save** before running the cells below.

The setup cells clone the `NLIstruct` repository (which contains the supporting Python modules), install dependencies, confirm CUDA is available, and pre-download NLTK corpora.

In [ ]:
# Clone the NLIstruct repo so the supporting modules are importable.
# Guarded so re-running the cell after a reconnect doesn't error.
![ -d nli-argument-structure ] || git clone https://github.com/seo-sangmin/nli-argument-structure.git
%cd nli-argument-structure
# !git checkout <branch-name>

In [ ]:
# Install dependencies pinned in requirements.txt.
!pip install -q -r requirements.txt

In [ ]:
# GPU sanity check. Expected: Tesla T4 in nvidia-smi and CUDA available: True.
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. The pipeline will fall back to CPU and run very slowly.")
    print("Select Runtime → Change runtime type → T4 GPU and rerun.")

In [ ]:
# Pre-download NLTK corpora used downstream. The preprocessing module also
# downloads these on import, but doing it explicitly here surfaces any
# network issue before the pipeline starts.
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

# 2. Analysing the Target Texts

Mirrors `run_text_analysis()` from `main.py`.

## 2.1 Pre-processing

Load, clean, and tokenize the two source texts (Chinese Room Argument and Minds, Brains, and Programs). The texts are fetched from Google Drive via `gdown` on first run. If a gdown rate-limit error appears, rerun this cell.

In [ ]:
from preprocessing import load_cra, load_mbp, count_text_stats

cra_raw, cra_clean, cra_tokens = load_cra()
mbp_raw, mbp_clean, mbp_tokens = load_mbp()

## 2.2 Exploratory Analysis

Count sentences, words, and unique words, then display word clouds of the most frequent terms in each text.

In [ ]:
from exploratory import print_text_stats, plot_wordclouds

cra_stats = count_text_stats(cra_raw)
mbp_stats = count_text_stats(mbp_raw)
print_text_stats("Chinese Room Argument (CRA)", *cra_stats)
print_text_stats("Minds, Brains, and Programs (MBP)", *mbp_stats)

plot_wordclouds(cra_tokens, mbp_tokens)

## 2.3 Topic Modelling

Use bag-of-words vectorization and Latent Dirichlet Allocation (LDA) to identify topics across both texts, and visualize each topic as a word cloud.

In [ ]:
from topic_modeling import run_lda, plot_topic_wordclouds

topics = run_lda([cra_clean, mbp_clean])
print("Identified topics:", topics)

plot_topic_wordclouds(topics)

## 2.4 Sentence Embedding

Use a Sentence-Transformer to embed each sentence, project the embeddings to 2D with t-SNE, and plot them with Plotly. Hover over points to inspect the underlying sentences.

*The `embedding` module sets the Plotly renderer automatically when running in Colab. If the plot shows blank (e.g. after reopening a saved notebook), just re-run this cell. If it was blank earlier in this same session, start a fresh runtime first — an earlier run may have left an incompatible Plotly version installed.*

In [ ]:
from embedding import embed_sentences, reduce_to_2d, plot_embeddings

cra_sentences, cra_embeddings = embed_sentences(cra_raw)
mbp_sentences, mbp_embeddings = embed_sentences(mbp_raw)

x_cra, y_cra = reduce_to_2d(cra_embeddings)
x_mbp, y_mbp = reduce_to_2d(mbp_embeddings)

plot_embeddings(x_cra, y_cra, cra_sentences, x_mbp, y_mbp, mbp_sentences)

## 2.5 Text Summarization

Summarize both texts with [LongT5](https://huggingface.co/pszemraj/long-t5-tglobal-base-16384-book-summary), a text-to-text model fine-tuned for long-sequence summarization (Raffel et al., 2019; Guo et al., 2022).

The model (~1 GB) is downloaded on first use and runs on GPU automatically.

In [ ]:
from summarization import create_summarizer, summarize_and_display

summarizer = create_summarizer()
wrapped_cra_sum, wrapped_mbp_sum = summarize_and_display(summarizer, cra_raw, mbp_raw)

# 3. Applying Natural Language Inference to Argument Analysis

Natural Language Inference (NLI) is a classification task that takes a premise-hypothesis pair as input and predicts one of three inference types: entailment, contradiction, or neutral. The Stanford NLI corpus (SNLI) and the Multi-NLI corpus (MNLI) are commonly used as training datasets for NLI tasks.

## 3.1 Fine-tuning BERT on SNLI

> **⚠️ OPTIONAL — this cell takes approximately 30–50 minutes on a T4 GPU.**
>
> Section 3.2 uses a different, pre-trained CrossEncoder (DeBERTa-v3) and does **not** depend on the model trained here. Skip to 3.2 unless you specifically want to reproduce the BERT fine-tuning result (~0.80 accuracy on SNLI).
>
> If you plan to run the full pipeline top-to-bottom, consider `Runtime → Restart` before this cell to free VRAM held by LongT5 from Section 2.5.

Fine-tune BERT on SNLI following the methodology from [*Dive into Deep Learning* §16.7](https://d2l.ai/chapter_natural-language-processing-applications/natural-language-inference-bert.html).

In [ ]:
# d2l 1.0.3 pins outdated sub-dependencies (e.g. gym==0.21.0) that fail to
# build on current Colab. Installing with --no-deps skips those pins;
# Colab already provides numpy/pandas/matplotlib/etc. that d2l needs at runtime.
!pip install -q --no-deps d2l==1.0.3

from bert_snli import load_pretrained_bert, get_devices, train_classifier

bert_model, _vocabulary = load_pretrained_bert()
devices = get_devices()
train_classifier(bert_model, devices)

## 3.2 Using an NLI Model for the Chinese Room Argument

Use a fine-tuned DeBERTa-v3 CrossEncoder (trained on SNLI + MNLI, 90.04% on MNLI topic-mismatched) to label the premise-conclusion relationships for four argument formulations: **MBP**, **CRA**, **LARG**, and **LARG_PARA**. See [Pre-trained Cross Encoders](https://www.sbert.net/docs/pretrained_cross-encoders.html#nli).

The combined DataFrame aggregates all four analyses; the bar chart compares model predictions to the expected (human) labels.

In [ ]:
from nli_analysis import load_nli_model, run_all_argument_analyses, plot_match_percentages

nli_model = load_nli_model()
combined_df = run_all_argument_analyses(nli_model)

print("\n--- Combined Results ---")
print(combined_df)

plot_match_percentages(combined_df)

# 4. Limits of Identifying the Argument Structure in Searle's Texts Using NLI

## 4.1 Limits of Labeling Argument Types

I will argue that, based on the results, the correct analysis of argument types is restricted to very clear arguments, such as LARG, at least when using the model I employed.

For arguments that have a complete premise, I assigned the label "entailment," and for all others, I assigned "neutral." These assignments align with human predictions for specific reasons. According to Korman et al. (2018), textual entailment is defined as follows: $T$ textually entails $H$ if "typically, a human reading $T$ would be justified in inferring the proposition expressed by $H$ from the proposition expressed by $T$." This definition avoids many counterexamples proposed by Korman et al.

Recognizing Textual Entailment (RTE) is a task similar to NLI. We can state that 'contradiction' occurs when $T$ textually entails $\neg H$, and 'neutral' occurs when $T$ neither textually entails $H$ nor $\neg H$.

Furthermore, this definition aligns with the labeling instructions for SNLI and MNLI. Annotators for these datasets were instructed to label the hypothesis as definitely true or correct (entailment), possibly true or correct (neutral), or definitely false or incorrect (contradiction), based on the premise (Bowman, 2015; Williams, 2017).

It is sufficient to judge NLI labels based on the definition of textual entailment. If a human reader would typically be justified in inferring $ H$ from $T$, then people will judge that $H$ is true based on $T$. Conversely, if $\neg H$ would typically be inferred from $T$, then $H$ would be judged as false or incorrect based on $T$. If neither $H$ nor $\neg H$ is typically inferred from $T$, then $H$ will be judged as possibly true or possibly false based on $T$. Therefore, we can apply the definition of textual entailment when making human predictions about analyzing argument structures.

Let's reconsider the summarized Chinese Room argument based on Searle's work from 1980.

Premise 1: As far as the Chinese is concerned, I simply behave like a computer. I have inputs and outputs that are indistinguishable from those of the native Chinese speaker, but I still understand nothing. (room_mbp)

Premise 2: In the Chinese case the computer is me, and in cases where the computer is not me, the computer has nothing more than I have in the case where I understand nothing. (com_mbp)

Conclusion: Computer understands nothing of any stories, whether in Chinese, English, or whatever. (conc_mbp)

If a computer does not understand Chinese, just as I don't in the Chinese Room, then it follows that the computer does not understand stories written in Chinese. Furthermore, if the computer lacks understanding, similar to how I lack understanding for other languages, then it can be concluded that computers do not understand stories written in any language. People would typically be justified in inferring 'conc_mbp' from 'room_mbp' and 'com_mbp'. While the arguments themselves may be false or debatable, their structural logic holds if we assume the premises to be true. This would categorize them as "entailments" in the context of NLI.

However, if one of the premises is missing, the conclusion 'conc_mbp' is not justified. For instance, if 'room_mbp' is not assumed, it is possible that I understand some languages, including Chinese. Therefore, the computer might also understand Chinese. This is because, although the computer is likened to me, there is no assumption that I am in the room processing Chinese without any understanding. In this case, neither 'conc_mbp' nor $\neg$'conc_mbp' is justified, as it is also plausible that the computer understands nothing. Therefore, the argument is neither an entailment nor a contradiction; it is neutral.

Similarly, if 'com_mbp' is not assumed, it's possible that the computer might understand some stories, as it could be more capable than I am in the room. Thus, 'conc_mbp' could still be true or false, making the argument neutral in this case as well. Other types of arguments, such as CRA, LARG, and LARG_PARA, can be analyzed in a similar manner.

LARG appears to have a more straightforward structure:

Premise 1: Implemented programs are by definition purely formal or syntactical. (prog)

Premise 2: Minds have mental or semantic contents. (mind)

Premise 3: Syntax is not by itself sufficient for, nor constitutive of, semantics. (suf)

Conclusion: Implemented programs are not constitutive of minds. (conc_larg)

Let $P$ represent '$x$ is an implemented program,' $Y$ represent '$x$ is by definition purely formal or syntactical,' $M$ represent '$x$ is a mind,' and $S$ represent '$x$ has mental or semantic contents.' Then, 'prog' is $P \rightarrow Y$, 'mind' is $M \rightarrow S$, 'suf' is $Y \rightarrow \neg S$, and 'conc_larg' is $P \rightarrow \neg M$. By the contrapositive of 'suf,' $S \rightarrow \neg Y$ holds. Similarly, by the contrapositive of 'prog,' $\neg Y \rightarrow \neg P$ holds. Thus, $S \rightarrow \neg P$, and by 'mind,' $M \rightarrow \neg P$. Therefore, $P \rightarrow \neg M$. Typically, a human would be justified in inferring 'conc_larg' from the conjunction of these premises, making the conclusion an entailment. If any premise is omitted, the conclusion is neither entailed nor contradicted. Paraphrased versions also maintain the same structure.

In [ ]:
from nli_analysis import plot_match_percentages

plot_match_percentages(combined_df)

As the plot shows, most predictions for labels do not align with human predictions, except for LARG. Furthermore, the human predictions for each conjunction of all premises to each conclusion do not match at all with the model predictions, while LARG's predictions are all consistent.

In [ ]:
from discussion import show_entailment_cases

entail_df = show_entailment_cases(combined_df)
entail_df

This discrepancy may be because LARG has a clearer structure and content than other arguments like LARG_PARA. Specifically, the term 'mind' is used instead of 'cognitive processes,' which lends greater clarity to the argument.

- 'mind': Minds have mental or semantic contents.
- 'mind_para': Human cognitive processes contain elements that are mental or semantic.

However, natural language arguments are generally less clear-cut than LARG. Therefore, we can conclude that NLI models, particularly the DeBERTa model which achieves approximately 90% accuracy on the NLI test dataset, are not effective at analyzing argument types found in 'cra' and 'mbp'.

In subsequent sections, I will argue that identifying the argument structure of 'cra' or 'mbp' requires the integration of additional complementary techniques, even if we assume the existence of a model that labels argument types correctly.

## 4.2 The Need for Argument Mining

Before analyzing the labels of arguments, it's essential to first identify the arguments and their components within texts like 'cra' or 'mbp'. For my argument analysis, I manually extracted the arguments and their individual components, such as premises and conclusions, as well as the relationships between these components (e.g., whether the conjunction of all premises entails the conclusion). This process is necessary even for clearly structured arguments like LARG, especially when embedded in a larger text like 'cra,' which contains 1024 words.

In [ ]:
cra_num_sentences, cra_num_words, _ = cra_stats
mbp_num_sentences, mbp_num_words, _ = mbp_stats

print(cra_num_words)
print(mbp_num_words)

Even if premises and conclusions seem clearly ordered and labeled to a human reader, they still need to be explicitly identified. Furthermore, it's also important to determine whether the combined premises entail the conclusion, particularly if multiple arguments are presented.



Some might suggest analyzing the entire text with Natural Language Inference (NLI). However, relying solely on NLI for natural language text is computationally inefficient. Let $n$ be the number of sentences in a natural language text. If NLI is correct, the inference from a premise to a conjunction of premises can be derived from the inferences from the premise to each sentence in the conjunction. Thus, when the premise consists of one sentence, the number of cases is $\binom{n}{1} \cdot \binom{n-1}{1}$. For a premise with two sentences, the number of cases is $\binom{n}{2} \cdot \binom{n-2}{1}$. (Note that inference from the conjunction of $n$ sentences to the conclusion is not derived from the inference from a single-sentence premise to the conclusion.) For a premise with $n-1$ sentences, the number of cases is $\binom{n}{n-1} \cdot \binom{1}{1}$. Therefore, NLI needs to be performed $\binom{n}{1}\cdot \binom{n-1}{1} + \binom{n}{2}\cdot \binom{n-2}{1} + \ldots + \binom{n}{n-1}\cdot \binom{1}{1} = \sum_{k=1}^{n-1} \binom{n}{k} \cdot \binom{n-k}{1}$ times.

This number grows significantly larger than \( 2^n \) as \( n \) increases, as illustrated in the graph below.

In [ ]:
from discussion import plot_nli_complexity

plot_nli_complexity()

Considering the number of sentences in 'cra' and 'mbp,' which are 59 and 352 respectively, the numbers \(2^{59}\) and \(2^{352}\) are already computationally inefficient.



In [ ]:
print(cra_num_sentences)
print(mbp_num_sentences)

What about reducing the number of sentences using text summarization techniques?

In [ ]:
print('Summary of cra')
print(wrapped_cra_sum)
print()
print('Summary of mbp')
print(wrapped_mbp_sum)

However, the results in Section 2.5 appear to be incorrect. For example, the summary of 'cra' states, "The argument against strong artificial intelligence goes something like this: An appropriately programmed, digital machine withthe right inputs/outputs, one who satisffies the test, would certainly have a brain." This assertion contradicts the original 'cra' argument, which does not claim that such machines would have brains. Additionally, none of the summaries for 'mbp' or 'cra' contain any sentences that resemble the main conclusions of the arguments—namely, 'conc_mbp,' 'conc_cra,' or 'conc_larg.'



In [ ]:
from discussion import print_conclusions

print_conclusions()

Even if we assume the existence of a model capable of accurate summarization, it may still omit some arguments. Therefore, it seems more appropriate to focus on improving and employing argument mining techniques for our purposes. The process of argument mining aligns well with our objective of identifying argument structures in 'mbp' and 'cra.' According to Lawrence & Reed (2020) and Ruiz-dolz et al. (2021), the task of argument mining consists of three sub-tasks:

1. Distinguishing arguments from non-arguments.
2. Identifying clausal properties, such as premises and conclusions.
3. Determining the relationships between argumentative propositions.

## 4.3 The Need for Probabilistic Analysis

What is another challenge in identifying argument structures from natural language texts like 'cra' and 'mbp' arises even when we have a model that labels argument types accurately? In many instances, argument analysis requires a probabilistic interpretation. For example, consider the following case from Chen et al. (2020):

- (Premise) A boy hits a ball with a bat.
- (Hypothesis) The boy is playing in a baseball game.

While this example is labeled as an "entailment" in the Stanford Natural Language Inference (SNLI) dataset, it could arguably be considered "neutral."

In [ ]:
from discussion import example_boy_ball

example_boy_ball(nli_model)

 A more specific label might be to assign a probability of around 0.9 to its being an entailment.

In the arguments we are examining, omitting a premise often leads to a "neutral" label.

In [ ]:
from discussion import example_room_mbp_partial

example_room_mbp_partial(nli_model)

In [ ]:
from discussion import example_larg_partial

example_larg_partial(nli_model)

(The model's prediction is also 'neutral' in this case.) However, labeling probabilities would be more specific because the conclusion will have a probability between 0 and 1, depending on the actual probability of the excluded premise. For example, if the probability of 'com_mbp' is actually high, the argument from 'room_mbp' to 'conc_mbp' would also have a high probability.

As can be seen, a more nuanced approach would be to evaluate it in terms of subjective probability. This is the idea behind "Uncertain NLI" (UNLI), as proposed by Chen et al. (2020). They created a dataset called u-SNLI by relabeling approximately 12% of the pairs in the original SNLI dataset. By fine-tuning BERT on both SNLI and u-SNLI, they achieved performance metrics close to human levels on development and test datasets. For instance, the transition from 'A man is singing into a microphone' to 'A man is performing on stage' is labeled as 84% in u-SNLI (and as 'neutral' in SNLI).

However, this does not mean that the model has human-level accuracy for all possible premise-hypothesis pairs. Additional UNLI data and experiments are needed to more fully evaluate probabilistic reasoning and inference.

## 4.4 The Need for Justification

Certain cases in Natural Language Inference (NLI) may require additional justification. Some inferences are straightforward for humans to grasp. For example, the statement 'Quine is married' contradicts the hypothesis 'Quine is a bachelor.'

In [ ]:
from discussion import example_quine

example_quine(nli_model)

However, there are instances where the model computes inferences as entailments or contradictions, and it's challenging to understand the reasoning behind such labels. For the argument type LARG, the model accurately predicted all labels except for two. While this could be attributed to a limitation of the model, it's not clear whether the human prediction (neutral) or the model's prediction (contradiction) is correct, especially if we generally assume the model to be accurate.





In [ ]:
from discussion import example_larg_orderings

example_larg_orderings(nli_model)

These kinds of inferences are akin to human intuitive reasoning, where explicit justification may not be readily available. NLI models don't operate with formal proofs or structured arguments. If we could establish the validity of an NLI model's learning and prediction processes, it would lend more credence to the inferences made by the model.

However, the architecture of Artificial Neural Networks (ANNs) used in these models, along with their "black box" nature, complicates any straightforward analysis. This raises further questions about the interpretability and explainability of such models.

The generation of additional arguments might provide the needed justification for certain inferences. For instance, consider an argument of the form "$p$, therefore $q$" that is labeled as 0.1, indicating a contradiction. If the rationale behind this label is unclear, introducing an intermediate premise $p'$ could shed light on the situation. If a model finds $p'$ such that the argument "$p$, therefore $p'$" is labeled as 1 (indicating entailment), and "$p'$, therefore $q$" is also labeled as 0.1 (indicating contradiction), then the chain of NLI through $p'$ could offer the required justification.

However, for this solution to work, NLI must first be accurate. Consider the following example: 'inter' appears to serve as $p'$, helping us understand why the argument from 'mind' + 'prog' to 'conc_larg' is a contradiction. This might be because 'programs with minds,' which possess semantics, could exist. (If it is possible.) 'inter' is entailed from 'mind' + 'prog' and contradicts 'conc_larg'.

Yet, the premise 'inter2,' which is semantically very similar to 'inter1,' is contradicted by 'mind' + 'prog' while also contradicting 'conc_larg.' This discrepancy makes it impossible to accept the argument either through 'inter' or 'inter2,' and it highlights the model's inaccuracies. These inaccuracies may be due to the model's interpretation of the phrases 'minds are mental and semantic' in 'inter1' and 'programs are mental and semantic' in 'inter2'.

In [ ]:
from discussion import example_intermediate

example_intermediate(nli_model)

# 5. Conclusion

I have analyzed texts from Searle (1980) and Searle (2001) using NLP techniques and applied NLI to specific arguments within these texts. The results and my subsequent analyses indicate that there are limitations to using NLI for identifying argumentative structures in such texts. I hope that future research will develop complementary techniques and justifications, thereby facilitating more accurate identification of argumentative structures in any given text.

# References

Bowman, S. R., Angeli, G., Potts, C., & Manning, C. D. (2015). A large annotated corpus for learning natural language inference. *arXiv preprint arXiv:1508.05326*.

Chen, T., Jiang, Z., Poliak, A., Sakaguchi, K., & Van Durme, B. (2019). Uncertain natural language inference. *arXiv preprint arXiv:1909.03042*.

Guo, M., Ainslie, J., Uthus, D., Ontañon, S., Ni, J., Sung, Y. H., & Yang, Y. (2022). LongT5: Efficient text-to-text transformer for long sequences. In *Findings of the Association for Computational Linguistics: NAACL 2022* (pp. 724–736).

Korman, D. Z., Mack, E., Jett, J., & Renear, A. H. (2018). Defining textual entailment. *Journal of the Association for Information Science and Technology*, 69(6), 763–772.

Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., ... & Liu, P. J. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer. *The Journal of Machine Learning Research*, 21(1), 5485–5551.

Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *arXiv preprint arXiv:1908.10084*.

Searle, J. R. (1980). Minds, brains, and programs. *Behavioral and Brain Sciences*, 3(3), 417–424.

Searle, J. R. (2001). Chinese room argument. In R. A. Wilson & F. C. Keil (Eds.), *The MIT Encyclopedia of the Cognitive Sciences*. MIT Press.

Williams, A., Nangia, N., & Bowman, S. R. (2017). A broad-coverage challenge corpus for sentence understanding through inference. *arXiv preprint arXiv:1704.05426*.

Zhang, A., Lipton, Z. C., Li, M., & Smola, A. J. (2023). *Dive into Deep Learning*.